# Function Call 01 · 工具 = 函数 + 给模型看的描述

整章 `04_function_call/` 的第一课，只回答一个问题：
**模型是怎么「用上」你写好的 Python 函数的？**

答案分两半，缺一不可：

| 概念 | 是什么 | 本节的代码形态 |
|---|---|---|
| 工具函数 | 真正干活的普通 Python 函数 | `tools_jxsd.py` 的 `add_tool` / `get_weather` |
| 工具描述 | 发给模型看的 JSON Schema（`name` / `description` / `parameters`） | `tool_desc_jxsd.py` 的 `TOOLS` |
| 对齐键 | **函数名** —— 两边靠它对齐 | `TOOL_REGISTRY["mul_tool"]` |
| 注册表 | 白名单：模型只能调到名单里的函数 | `TOOL_REGISTRY` |
| 分发器 | 按模型报出的名字取出真函数并执行 | `call_tool("mul_tool", a=4, b=6)` |

一句话记住这件事：**模型看不到函数代码，它只看到一份描述；
真正执行的是你的 Python 函数 —— 两边靠函数名对齐。**

课案《概念》小节的两句原话（整章的总纲）：

> 「Function Call 是大模型调用外部工具的标准化协议，现在改名为 Tool Call。」
>
> 「Function Call 允许大模型根据用户输入自动决定是否需要调用外部工具，
> 并生成相应的函数调用。」

拆开看，里面藏着三个「谁」：

| 问题 | 答案 |
|---|---|
| 谁决定 | **大模型自己**（不是我们写 `if/else` 判断） |
| 决定什么 | 调哪个工具、参数填什么（以结构化 JSON 给出，不是文字描述） |
| 谁执行 | **本节的 Python 函数**（模型只写「申请书」，真正干活的是程序） |

> **本 notebook 由 `Agent/04_function_call/` 下 4 个脚本合并而成**：
> `tools.py`（课案原版 47 行）、`tool_desc.py`（课案原版 51 行）、
> `tools_jxsd.py`（完整版 379 行）、`tool_desc_jxsd.py`（完整版 455 行）。
> 前两个是「最短实现」，后两个是「完整版」—— 先看最短的，再看完整版差在哪。
>
> ⚠️ 这四个文件原本都是**被别人 import 的库**（`02_三种Agent实现对比.ipynb` 就 import 它们），
> 所以本节原样保留全部函数与常量定义，只把原来 `if __name__ == "__main__":` 里
> 的演示段顶格写出来。下一课会把这里的工具真发给模型。

**官方文档**
- 工具（Tools）：<https://docs.langchain.com/oss/python/langchain/tools>
- 智能体（Agents）：<https://docs.langchain.com/oss/python/langchain/agents>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟢 运行档位 | **离线可跑** —— 不读 `.env`、不连大模型、不起任何服务、不花钱 |
| 依赖 | **只用标准库 `json`**（本节不 import 任何框架、不 import `config`） |
| 密钥 | 不需要 |
| 前置服务 | 无 |
| 预计耗时 | < 1 秒 |
| 语言要求 | Python ≥ 3.10（`tools_jxsd.py` 用了 `list[str]` 与 `None` 的联合类型写法） |

> 这是 `04_function_call/` 这一章里**唯一零依赖、零配置、零费用**的一课，
> 可以放心反复跑。真调模型、真要 API Key 的是下一课 `02_三种Agent实现对比.ipynb`。

## 本节地图

先看一次完整的 Function Call 回合里，数据在「你」和「模型」之间怎么走 ——
**本节只实现其中 ④⑤ 两步（工具与描述）**，②③⑥ 要等下一课接上模型才能真跑：

```mermaid
graph LR
    D["工具描述 TOOLS<br/>name / description / parameters"] -.->|"① 随请求一起发"| M["大模型"]
    U["用户提问"] -.->|"② 一起发过去"| M
    M -.->|"③ 只回一份申请书 tool_calls<br/>name + arguments JSON"| P["你的程序"]
    P -->|"④ 按名字查白名单"| R["TOOL_REGISTRY"]
    R -->|"⑤ 取出真函数并执行"| F["add_tool / get_weather / ..."]
    F -.->|"⑥ 结果拼回 messages 再发一轮"| M
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 步骤 | 谁做 | 本节的对应物 | 在哪一节 |
|---|---|---|---|
| ① 把工具描述随请求发给模型 | 你的程序 | `TOOLS` 列表 | 2.9 |
| ② 用户提问一起发过去 | 你的程序 | `messages` | 下一课 |
| ③ 模型决定调哪个工具、参数填什么 | **大模型** | `tool_calls` | 下一课 |
| ④ 按名字查白名单找真函数 | 你的程序 | `TOOL_REGISTRY` / `call_tool` | 2.3 / 2.6 |
| ⑤ 执行函数、拿到返回值 | Python 运行时 | `add_tool` … | 2.1 / 2.2 |
| ⑥ 把结果拼回 `messages` 再发一轮 | 你的程序 | 工具结果字符串 | 下一课 |

**与上下节的衔接**

- 上一节：`03_deepagents/` 结束，我们已经能「让 Agent 自己干活」；
  但它干的活全是模型自带的（写文件、跑 shell）—— 想让它调**你的**业务函数，就得先有工具。
- 本节：把工具**做成两件东西** —— 函数（给程序）+ 描述（给模型）。
- 下一节 `02_三种Agent实现对比.ipynb`：同一批工具，分别用
  **原生 OpenAI SDK 手写主循环** / **LangChain `@tool` + `create_agent`** /
  **DeepAgents `create_deep_agent`** 三种方式接上模型，对比「谁替你做了第 ①②③⑥ 步」。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

> 本课其实用不到 `config`（全程离线），但这一格仍然保留 ——
> 一是保持全仓统一，二是它顺便给出了 `NB_DIR` / `WORKDIR` 两个变量。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

引导格自己会打印仓库根与临时目录（路径随你把仓库放在哪里而变），输出不固定，故不写预期输出。

下面这格做**前置条件自检**：本课零依赖、零配置，唯一的硬要求是
**Python ≥ 3.10** —— 因为 `tools_jxsd.py` 用了 `list[str] | None`
这种联合类型写法（3.9 及以前会 `TypeError`）。
不满足时打印中文提示，让你知道该换解释器，而不是盯着后面的 `TypeError` 发懵。

In [ ]:
# 前置条件自检：本课只用标准库，唯一硬要求是 Python ≥ 3.10
PREREQ_OK = sys.version_info >= (3, 10)
print("Python 版本：", sys.version.split()[0], "| 满足 ≥3.10：", PREREQ_OK)
if PREREQ_OK:
    print("前置条件满足：本课只用标准库 json，不需要 .env / API Key / 数据库 / 端口")
else:
    print("[跳过] 本课需要 Python 3.10 及以上，请改用仓库 .venv 里的解释器再跑")

### 预期输出

```text
Python 版本： 3.12.12 | 满足 ≥3.10： True
前置条件满足：本课只用标准库 json，不需要 .env / API Key / 数据库 / 端口
```

## 1. 课案原版：最短实现（tools.py 47 行 + tool_desc.py 51 行）

课案原版把这件事拆成两个文件，各自只干一件事：

| 文件 | 给谁看 | 内容 |
|---|---|---|
| `tools.py` | **给程序看** | 两个普通 Python 函数 + 一张注册表 |
| `tool_desc.py` | **给模型看** | 两份 JSON 描述 + 一个列表 `TOOLS` |

先把最短的实现看完，再看完整版差在哪 —— 这个对比就是本课要讲的全部内容。

### 1.1 工具本体：工具首先得是一个「能按名字调用的函数」

`tools.py` 里没有任何框架痕迹：**没有装饰器、没有类型注解、没有 `import` 框架**。
这就是「工具」最朴素的形态 —— 框架提供的 `@tool` 装饰器，
干的事也只是帮你**自动生成下一节那份描述**而已。

In [ ]:
# ================================================================
# 一、课案原版 tools.py：工具函数 + 注册表
# ================================================================
import json
import random


# ---------- 普通函数版（给原生 OpenAI SDK 用） ----------
def get_weather(city: str) -> str:
    """
    查询指定城市的天气。

    :param city: 城市名称，如「上海」
    :return: 天气描述字符串
    """
    weather_map = {
        "上海": ("晴", 25),
        "北京": ("多云", 18),
        "广州": ("阵雨", 30),
    }
    desc, temp = weather_map.get(city, ("未知", random.randint(0, 35)))
    return json.dumps({"city": city, "weather": desc, "temperature": temp},
                      ensure_ascii=False)


def get_current_time() -> str:
    """获取当前时间"""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


# 工具注册表：模型返回工具名后，程序在这里查到真正要执行的函数
TOOL_REGISTRY = {
    "get_weather": get_weather,
    "get_current_time": get_current_time,
}

两个细节值得停一下：

1. `get_weather` 的返回是 **JSON 字符串**而不是 `dict` —— 因为工具结果最终要拼进
   `messages` 的 `content` 字段，那里只接受字符串；
2. 表外的城市走 `weather_map.get(city, ("未知", random.randint(0, 35)))` 兜底，
   所以温度是**随机的**（本课只查表内的「上海」，输出才稳定）。

### 1.2 工具描述：模型真正「看」到的东西

`tool_desc.py` 把工具**翻译**成模型能读懂的 JSON。OpenAI 格式是三层，三要素各管一件事：

| 层级 | 键 | 作用 |
|---|---|---|
| 第一层 | `type` | 固定 `"function"`，表示这是一个函数工具 |
| 第二层 | `function.name` | 函数名 —— 模型据此**指名调用** |
| 第二层 | `function.description` | 干什么用 —— 模型据此**判断何时该调用** |
| 第三层 | `function.parameters` | 参数的 JSON Schema —— 模型据此**填参** |

第二层那两件事分得很清楚：
**`name` 决定「调谁」，`description` 决定「什么时候想起来调它」**。
少写一句 `description`，模型就可能永远想不起来该用这个工具。

In [ ]:
# ================================================================
# 二、课案原版 tool_desc.py：工具描述（JSON Schema）
# ================================================================
# 查天气工具的描述（与 tools.get_weather 对应）
GET_WEATHER_DESC = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "查询指定城市的实时天气，包括天气状况和温度",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "城市名称，例如：上海",
                },
            },
            "required": ["city"],
        },
    },
}

# 查时间工具的描述（无参数）
GET_TIME_DESC = {
    "type": "function",
    "function": {
        "name": "get_current_time",
        "description": "获取当前的日期和时间",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
        },
    },
}

# 发给 API 的 tools 参数：工具描述列表
TOOLS = [GET_WEATHER_DESC, GET_TIME_DESC]

注意 `GET_TIME_DESC` 这个「无参数工具」的写法：`properties` 是空字典、
`required` 是空列表 —— **不是省略掉 `parameters`**。协议要求这一层永远存在，
只是可以为空。

### 1.3 最小闭环：模型只写「申请书」，干活的是 Python 函数

这两个文件原本都是库、没有 `if __name__ == "__main__":` 演示段，
这里补一个最小闭环，把「工具描述 → 模型决定 → 程序执行」这条链手工走一遍。

⚠️ 下面那个 `tool_call` 是**手工模拟模型返回值**（本课不联网）。
真实场景里它是 `message.tool_calls[0]`，结构一模一样：
`name` 是函数名，`arguments` 是**字符串形式的 JSON**（不是 dict，得先 `json.loads`）。

In [ ]:
# ================================================================
# 三、课案原版的最小闭环（教材补的演示段）
# ================================================================
print("=" * 62)
print("课案原版：tools.py + tool_desc.py 的最小闭环")
print("=" * 62)
print("发给模型的工具清单：", [t["function"]["name"] for t in TOOLS])
print()
print("模型看到的 get_weather 描述：")
print(json.dumps(GET_WEATHER_DESC, ensure_ascii=False, indent=2))
print()

# 模拟模型返回的 tool_call：name 指名调谁，arguments 是 JSON 字符串
tool_call = {"name": "get_weather", "arguments": '{"city": "上海"}'}
print("模拟模型返回的 tool_call：", tool_call)

# 程序这边做两件事：把 arguments 解析成 dict，再按 name 从注册表取真函数
result = TOOL_REGISTRY[tool_call["name"]](**json.loads(tool_call["arguments"]))
print("程序按名字查注册表 → 执行真函数 → 工具结果：", result)
print()
print("注册表里的名字：", list(TOOL_REGISTRY))
print("get_current_time() →", get_current_time())

### 预期输出

```text
==============================================================
课案原版：tools.py + tool_desc.py 的最小闭环
==============================================================
发给模型的工具清单： ['get_weather', 'get_current_time']

模型看到的 get_weather 描述：
{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "查询指定城市的实时天气，包括天气状况和温度",
    "parameters": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "城市名称，例如：上海"
        }
      },
      "required": [
        "city"
      ]
    }
  }
}

模拟模型返回的 tool_call： {'name': 'get_weather', 'arguments': '{"city": "上海"}'}
程序按名字查注册表 → 执行真函数 → 工具结果： {"city": "上海", "weather": "晴", "temperature": 25}

注册表里的名字： ['get_weather', 'get_current_time']
get_current_time() → 2026-09-17 15:37:47
```

**三个值得看的点：**

1. `TOOL_REGISTRY[tool_call["name"]]` 这一句就是「对齐键」的全部意义：
   模型给的是字符串 `"get_weather"`，程序靠它**在字典里查到函数对象**；
2. 工具结果是**字符串**（`json.dumps` 出来的），可以直接塞进 `messages` 的 `content`；
3. 最后一行的时间戳**每次运行都不同** —— 上面那个值只是我这次跑出来的实测值。

## 2. 完整版：把 8 种参数类型逐个落成真函数

课案原版的 `tools.py` 只有 2 个工具，而且 `tool_desc.py` 的 `parameters` 是**每份描述各写一遍**。
课案《参数类型》小节另外给出了 8 个 JSON Schema 片段，但**只有片段、没有函数** ——
学员看不出「Schema 的 `string` 到 Python 这边到底是什么类型」。

完整版（`tools_jxsd.py` + `tool_desc_jxsd.py`）补齐了这条对照关系，
并且把 `parameters` 改成**按工具名查表**，工具扩容到 13 个也不用改主循环。

| | 课案原版 | 完整版 |
|---|---|---|
| 工具数量 | 2 个（`get_weather` / `get_current_time`） | 13 个（4 个数学 + 9 个参数类型示范） |
| 参数类型 | 只有 `string` 一种 | `string` / `number` / `integer` / `boolean` / `array` / `object` / `null` / `enum` |
| `parameters` 来源 | 每份描述手写一遍 | `TOOL_PARAMETERS` 按名字查表 |
| 查表分发 | `TOOL_REGISTRY[name]`（裸字典） | 加**名字白名单 + 执行兜底**两层保护 |

### 2.1 课案《实例》里的数学四则工具

课案原话：「下面是一个完整的实战案例，展示如何使用 Function Call 实现一个数学计算 Agent。」

这 4 个函数**刻意不带类型注解、不带 `@tool` 装饰器**，就是最普通的 Python 函数 ——
目的是先把最朴素的形态讲清楚：工具本质上只是「一个能被程序按名字调用的函数」。

In [ ]:
# ================================================================
# 四、课案《实例》里的数学四则工具（课案原文，一字未改）
# ================================================================
def add_tool(a, b):
    """
    返回a+b的结果
    :param a:第一个数字
    :param b:第二个数字
    :return: a+b的结果
    """
    return a + b


def sub_tool(a, b):
    """
    返回a-b的结果
    :param a:第一个数字
    :param b:第二个数字
    :return: a-b的结果
    """
    return a - b


def mul_tool(a, b):
    """
    返回a*b的结果
    :param a:第一个数字
    :param b:第二个数字
    :return: a*b的结果
    """
    return a * b


def div_tool(a, b):
    """
    返回a/b的结果
    :param a:第一个数字
    :param b:第二个数字
    :return: a/b的结果
    """
    return a / b

### 2.2 参数类型：把 8 种 JSON Schema 类型对上 Python 形参

课案《参数类型》小节的开头原话：「OpenAI 的 tools 参数使用 JSON Schema 来定义函数参数，支持以下类型」。
这句话里有两个信息点：

1. 参数用的是 **JSON Schema 这套公开标准**，不是 OpenAI 自创格式 ——
   所以 `type` / `properties` / `required` / `items` / `enum` 的含义都可以去 JSON Schema 规范里查；
2. 「支持以下类型」= 模型能被**约束**到的取值范围。类型写得越准，模型乱填的概率越低
   （写 `number` 它会给你 `4.5`，写 `integer` 它才只给 `4`）。

**参数类型对照表（本课最重要的一张表）**：

| JSON Schema 类型 | 课案片段常量 | 落到哪个真函数 | Python 这边收到什么 |
|---|---|---|---|
| `string` | `SCHEMA_STRING` | `format_username_tool(username: str)` | `str` |
| `number` | `SCHEMA_NUMBER` | `calc_price_tool(price: float)` | `int` 或 `float` |
| `integer` | `SCHEMA_INTEGER` | `check_stock_tool(count: int)` | `int` |
| `boolean` | `SCHEMA_BOOLEAN` | `toggle_feature_tool(is_enabled: bool)` | `bool` |
| `array` | `SCHEMA_ARRAY`（**必须带 `items`**） | `summarize_tags_tool(tags: list[str])` | `list` |
| `object` | `SCHEMA_OBJECT` / `SCHEMA_OBJECT_NESTED` | `describe_user_tool(user: dict)` | `dict` |
| `enum` | `SCHEMA_ENUM_UNIT` | `convert_temperature_tool(value, unit)` | 取值被限定的 `str` |
| `["string", "null"]` | `SCHEMA_NULL` | `set_nickname_tool(optional_field=None)` | `str` 或 `None` |
| 多类型组合 | `SCHEMA_CREATE_USER` | `create_user_tool(...)` | 多个不同类型形参 |

**为什么每个函数都带类型注解？** 因为注解**不是给 Python 看的**（运行时压根不校验它）。
真正的用途在 `agent_langchain_jxsd.py`：`@tool` 装饰器靠
**形参名 + 注解 + docstring** 自动生成那份 JSON Schema —— 注解就是 `properties` 里的 `type`。
所以本文件的注解和 `tool_desc_jxsd.py` 里手写的 Schema 是**同一件事的两种写法**。

命名沿用课案的工具后缀约定：一律以 `_tool` 结尾。
`list_tools()` 正是靠这个后缀（`dir()` + `endswith("_tool")`）自动扫出全部工具的 ——
⚠️ 反过来说，**任何以 `_tool` 结尾的顶层名字都会被自动当成工具**发出去。

先看前四种：`string` / `number` / `integer` / `boolean`。

In [ ]:
# ================================================================
# 五、课案《参数类型》小节：8 种 JSON Schema 类型对应的真实函数
# ================================================================

# ---------- 2.1 string：字符串类型 ----------
def format_username_tool(username: str) -> str:
    """
    规范化用户名：去掉首尾空格、转小写、首字母大写
    :param username: 用户名（string 类型）
    :return: 规范化后的用户名
    """
    # 真实业务里「规范化」比「原样返回」更能看出参数确实传进来了
    return username.strip().lower().capitalize()


# ---------- 2.2 number：数字类型（整数或浮点数） ----------
def calc_price_tool(price: float) -> str:
    """
    计算商品的含税价格（税率 13%）
    :param price: 商品价格（number 类型，整数或小数都可以）
    :return: 含税价格的文字说明
    """
    total = round(price * 1.13, 2)
    return f"价格 {price} 元，含税（13%）后 {total} 元"


# ---------- 2.3 integer：整数类型 ----------
def check_stock_tool(count: int) -> str:
    """
    检查库存是否充足
    :param count: 商品数量（integer 类型，只接受整数）
    :return: 库存状态说明
    """
    if count >= 10:
        return f"库存 {count} 件，充足"
    return f"库存 {count} 件，偏少，建议补货"


# ---------- 2.4 boolean：布尔值 ----------
def toggle_feature_tool(is_enabled: bool) -> str:
    """
    开启或关闭某个功能开关
    :param is_enabled: 是否启用（boolean 类型，只能是 true / false）
    :return: 开关状态说明
    """
    return f"功能已{'开启' if is_enabled else '关闭'}"

接着是三种「结构化」类型：`array`（数组）、`object`（对象）、`null`（空值）。

- **`array` 必须声明 `items`**，否则模型不知道里面该放字符串还是对象；
  到 Python 这边就是一个 `list`。
- **`object` 必须声明 `properties`**，到 Python 这边就是 `dict`。
  ⚠️ 取值务必用 `.get` 兜底：模型偶尔会漏字段，直接 `user["age"]` 会
  `KeyError` 把整个主循环炸掉。
- **`null`** 在 JSON Schema 里的写法是 `"type": ["string", "null"]`（列表 = 多选一），
  到 Python 这边就是 `str | None`。

In [ ]:
# ---------- 2.5 array：数组类型（需指定 items 元素类型） ----------
def summarize_tags_tool(tags: list[str]) -> str:
    """
    统计标签：去重后按字典序输出
    :param tags: 标签列表（array 类型，元素是 string）
    :return: 标签统计结果
    """
    unique = sorted(set(tags))
    return f"共传入 {len(tags)} 个标签，去重后 {len(unique)} 个：{'、'.join(unique)}"


# ---------- 2.6 object：对象类型（properties 里再套 properties 就是嵌套对象） ----------
def describe_user_tool(user: dict) -> str:
    """
    把用户信息渲染成一句话（user 里可以带嵌套的 address 对象）
    :param user: 用户信息对象，形如 {"name": "张三", "age": 28,
                 "address": {"city": "南昌", "zip": "330000"}}
    :return: 用户描述
    """
    # 对象参数到了 Python 这边就是 dict，取值时务必用 .get 兜底：
    # 模型偶尔会漏字段，直接 user["age"] 会 KeyError 把主循环炸掉
    name = user.get("name", "匿名用户")
    age = user.get("age", "未知")
    address = user.get("address") or {}
    city = address.get("city", "未填写")
    zip_code = address.get("zip", "未填写")
    return f"{name}，{age} 岁，所在城市 {city}，邮编 {zip_code}"


# ---------- 2.7 null：空值（常用于可选字段） ----------
def set_nickname_tool(optional_field: str | None = None) -> str:
    """
    设置昵称，允许传空（null）
    :param optional_field: 昵称，可以为 null（type 写成 ["string", "null"]）
    :return: 设置结果说明
    """
    if optional_field is None:
        return "昵称已清空（模型传入的是 null）"
    return f"昵称已设置为：{optional_field}"

最后两种：`enum`（枚举，把取值限死在几个选项里）和多类型组合的 `create_user`。

`enum` 的作用就是**把取值限定在固定几个里**，模型只能从中挑一个。
但函数这边**仍然要按普通字符串处理**，不能假设模型一定守规矩 ——
所以 `convert_temperature_tool` 留了最后一个 `return` 兜底「不认识的单位」。

`create_user_tool` 对应课案《参数类型 → 8. 完整示例》里那段 `create_user`，
它把 `string` / `integer` / `boolean` / `array` 拼在一个函数里，
返回 **JSON 字符串**（理由同 1.1：工具结果要拼进 `content`，那里只收字符串）。

In [ ]:
# ---------- 2.8 enum：枚举（课案 agent.py 里那处 enum 片段的正确用法） ----------
def convert_temperature_tool(value: float, unit: str = "celsius") -> str:
    """
    温度单位换算（摄氏度 <-> 华氏度）
    :param value: 温度数值（number 类型）
    :param unit: 传入数值的单位，只能是 "celsius" 或 "fahrenheit"（enum 枚举，string 类型）
    :return: 换算结果
    """
    # enum 的作用就是**把取值限定在固定几个里**，模型只能从中挑一个；
    # 函数这边仍然要按普通字符串处理，不能假设模型一定守规矩
    if unit == "celsius":
        return f"{value}°C = {round(value * 9 / 5 + 32, 1)}°F"
    if unit == "fahrenheit":
        return f"{value}°F = {round((value - 32) * 5 / 9, 1)}°C"
    return f"不认识单位 {unit}，只支持 celsius / fahrenheit"


# ---------- 2.9 课案《8. 完整示例》的 create_user，落成真函数 ----------
def create_user_tool(name: str, email: str, age: int = 0,
                     is_active: bool = True, tags: list[str] | None = None) -> str:
    """
    创建用户（对应课案《参数类型 → 8. 完整示例》里的 create_user）
    :param name: 用户名（string）
    :param email: 邮箱（string）
    :param age: 年龄（integer）
    :param is_active: 是否激活（boolean）
    :param tags: 标签（array，元素是 string）
    :return: 创建结果的 JSON 字符串
    """
    user = {
        "name": name,
        "email": email,
        "age": age,
        "is_active": is_active,
        "tags": tags or [],
    }
    # 返回 JSON 字符串而不是 dict：工具结果最终要拼进 messages 的 content 字段，
    # 那里只接受字符串
    return json.dumps({"已创建用户": user}, ensure_ascii=False)

### 2.3 工具注册表：模型报出函数名后，程序靠它找到真函数

课案原话（关键点说明 3）：「正确处理 `message.tool_calls`，记录到历史并调用实际工具」。

白名单式注册表除了「好查」，还有一个**安全作用**：
模型只能调到这里列出的函数，不可能通过伪造名字让程序执行任意代码
（比如 `getattr` 到 `os.system`）。

注册表分成两张名单，是为了让不同的课案小节各取所需：

| 常量 | 内容 | 谁在用 |
|---|---|---|
| `MATH_TOOL_NAMES` | 4 个数学工具 | 课案《实例》那一节 |
| `PARAM_TOOL_NAMES` | 9 个参数类型示范工具 | 课案《参数类型》那一节 |
| `TOOL_REGISTRY` | 13 个「名字 → 函数对象」 | `call_tool` 分发 |

In [ ]:
# ================================================================
# 六、工具注册表：模型报出函数名后，程序靠它找到真函数
# ================================================================
# 数学四则工具：课案的「实例」只用这 4 个
MATH_TOOL_NAMES = ["add_tool", "sub_tool", "mul_tool", "div_tool"]

# 参数类型示范工具：课案《参数类型》小节那 8 种类型 + enum
PARAM_TOOL_NAMES = [
    "format_username_tool",
    "calc_price_tool",
    "check_stock_tool",
    "toggle_feature_tool",
    "summarize_tags_tool",
    "describe_user_tool",
    "set_nickname_tool",
    "convert_temperature_tool",
    "create_user_tool",
]

TOOL_REGISTRY = {
    "add_tool": add_tool,
    "sub_tool": sub_tool,
    "mul_tool": mul_tool,
    "div_tool": div_tool,
    "format_username_tool": format_username_tool,
    "calc_price_tool": calc_price_tool,
    "check_stock_tool": check_stock_tool,
    "toggle_feature_tool": toggle_feature_tool,
    "summarize_tags_tool": summarize_tags_tool,
    "describe_user_tool": describe_user_tool,
    "set_nickname_tool": set_nickname_tool,
    "convert_temperature_tool": convert_temperature_tool,
    "create_user_tool": create_user_tool,
}

### 2.4 本地自测：先把「函数本身没问题」验证掉

这一格就是 `tools_jxsd.py` 原来 `if __name__ == "__main__":` 里的内容，**顶格照抄**。

它的意义在于**分离故障**：先把函数验证掉，后面主循环出问题时才能确定是
「模型没调工具」而不是「函数写错了」。

In [ ]:
# ================================================================
# 七、本地自测：把每个工具都真调一遍（原 __main__ 顶格）
# ================================================================
print("=" * 62)
print("一、课案《实例》的数学四则工具")
print("=" * 62)
print(f"add_tool(2, 3)   -> {add_tool(2, 3)}")      # 5
print(f"sub_tool(10, 4)  -> {sub_tool(10, 4)}")     # 6
print(f"mul_tool(4, 6)   -> {mul_tool(4, 6)}")      # 24
print(f"div_tool(24, 6)  -> {div_tool(24, 6)}")     # 4.0

print()
print("=" * 62)
print("二、课案《参数类型》的 8 种类型 → 真实函数")
print("=" * 62)
print(f"[string ] format_username_tool('  ZHANGsan ') -> {format_username_tool('  ZHANGsan ')}")
print(f"[number ] calc_price_tool(100)                 -> {calc_price_tool(100)}")
print(f"[integer] check_stock_tool(3)                  -> {check_stock_tool(3)}")
print(f"[boolean] toggle_feature_tool(True)            -> {toggle_feature_tool(True)}")
print(f"[array  ] summarize_tags_tool(['b', 'a', 'b']) -> {summarize_tags_tool(['b', 'a', 'b'])}")
print(f"[object ] describe_user_tool(嵌套 address)     -> "
      f"{describe_user_tool({'name': '张三', 'age': 28, 'address': {'city': '南昌', 'zip': '330000'}})}")
print(f"[null   ] set_nickname_tool(None)              -> {set_nickname_tool(None)}")
print(f"[enum   ] convert_temperature_tool(25, 'celsius') -> {convert_temperature_tool(25, 'celsius')}")
print(f"[多类型 ] create_user_tool(...)                -> "
      f"{create_user_tool('张三', 'zhangsan@example.com', 28, True, ['学生', '篮球'])}")

print()
print("=" * 62)
print("三、工具注册表 TOOL_REGISTRY")
print("=" * 62)
print(f"数学工具：{MATH_TOOL_NAMES}")
print(f"参数示范工具：{PARAM_TOOL_NAMES}")
print(f"注册表共 {len(TOOL_REGISTRY)} 个工具，"
      f"名字与函数是否全部对齐：{all(globals()[n] is f for n, f in TOOL_REGISTRY.items())}")
print()
print("下一步：看 tool_desc_jxsd.py（工具描述 / JSON Schema），再看 agent_openai_jxsd.py（主循环）。")

### 预期输出

```text
==============================================================
一、课案《实例》的数学四则工具
==============================================================
add_tool(2, 3)   -> 5
sub_tool(10, 4)  -> 6
mul_tool(4, 6)   -> 24
div_tool(24, 6)  -> 4.0

==============================================================
二、课案《参数类型》的 8 种类型 → 真实函数
==============================================================
[string ] format_username_tool('  ZHANGsan ') -> Zhangsan
[number ] calc_price_tool(100)                 -> 价格 100 元，含税（13%）后 113.0 元
[integer] check_stock_tool(3)                  -> 库存 3 件，偏少，建议补货
[boolean] toggle_feature_tool(True)            -> 功能已开启
[array  ] summarize_tags_tool(['b', 'a', 'b']) -> 共传入 3 个标签，去重后 2 个：a、b
[object ] describe_user_tool(嵌套 address)     -> 张三，28 岁，所在城市 南昌，邮编 330000
[null   ] set_nickname_tool(None)              -> 昵称已清空（模型传入的是 null）
[enum   ] convert_temperature_tool(25, 'celsius') -> 25°C = 77.0°F
[多类型 ] create_user_tool(...)                -> {"已创建用户": {"name": "张三", "email": "zhangsan@example.com", "age": 28, "is_active": true, "tags": ["学生", "篮球"]}}

==============================================================
三、工具注册表 TOOL_REGISTRY
==============================================================
数学工具：['add_tool', 'sub_tool', 'mul_tool', 'div_tool']
参数示范工具：['format_username_tool', 'calc_price_tool', 'check_stock_tool', 'toggle_feature_tool', 'summarize_tags_tool', 'describe_user_tool', 'set_nickname_tool', 'convert_temperature_tool', 'create_user_tool']
注册表共 13 个工具，名字与函数是否全部对齐：True

下一步：看 tool_desc_jxsd.py（工具描述 / JSON Schema），再看 agent_openai_jxsd.py（主循环）。
```

**最后那行 `True` 分量最重**：它证明注册表里 13 个名字与函数对象
**全部对齐**（没写错名字、没漏注册）。名字对不上是 Function Call 最经典的故障 ——
现象是「模型说调用 `get_weather`，程序回一句『没有名为 get_weather 的工具』」。

### 2.5 让 `import tools_jxsd` 在 notebook 里成立

到这里有个**必须解决的结构问题**：`tool_desc_jxsd.py` 的第一行是 `import tools_jxsd`，
因为原世界里「函数」和「描述」是两个独立的**模块文件**。

可 notebook 里没有第二个模块文件，工具函数全都定义在**本内核的全局命名空间**里。
解法是：**把这批工具注册成一个名叫 `tools_jxsd` 的模块对象**放进 `sys.modules`
（`import` 的第一步就是查 `sys.modules`，命中就直接返回，不去文件系统找文件）。
这样下面那句 `import tools_jxsd` 原样可用，`dir(tools_jxsd)` 与
`getattr(tools_jxsd, name)` 也全部照旧工作 —— **源文件里一行都不用改**。

⚠️ **但不能图省事直接把 notebook 的 `__main__` 注册成 `tools_jxsd`**。
原因很刁钻，而且恰好命中源文件自己警告过的那条坑：
`list_tools()` 是用 `dir()` + `endswith("_tool")` 扫工具的，
而**`call_tool` 这个名字正好以 `_tool` 结尾**！

在原世界里 `call_tool` 定义在 `tool_desc_jxsd.py`，`dir(tools_jxsd)` 根本看不到它；
一旦两个模块被合并进同一个命名空间，它就会变成**第 14 个「工具」**
一起发给模型（实际跑出来就是这样，见下面注释里的对照）。
所以这里按**原模块的边界**挑名字，造一个只装这些名字的模块。

In [ ]:
# ================================================================
# 八、把工具注册成 tools_jxsd 模块（还原原来的模块边界）
# ================================================================
import types

# 只搬运「原 tools_jxsd.py 里确实存在的名字」：13 个工具函数 + 三张名单。
# 不搬 list_tools / call_tool —— 它们原本住在 tool_desc_jxsd.py，
# 并且 call_tool 的名字以 _tool 结尾，混进来就会被 list_tools() 当成工具扫走。
tools_jxsd = types.ModuleType("tools_jxsd")
for _name, _obj in list(globals().items()):
    if _name.endswith("_tool") or _name in ("MATH_TOOL_NAMES", "PARAM_TOOL_NAMES", "TOOL_REGISTRY"):
        setattr(tools_jxsd, _name, _obj)
sys.modules.setdefault("tools_jxsd", tools_jxsd)

import tools_jxsd

print("tools_jxsd 指向：", tools_jxsd)
print("它扫到的工具函数：", [n for n in dir(tools_jxsd) if n.endswith("_tool")])

### 预期输出

```text
tools_jxsd 指向： <module 'tools_jxsd'>
它扫到的工具函数： ['add_tool', 'calc_price_tool', 'check_stock_tool', 'convert_temperature_tool', 'create_user_tool', 'describe_user_tool', 'div_tool', 'format_username_tool', 'mul_tool', 'set_nickname_tool', 'sub_tool', 'summarize_tags_tool', 'toggle_feature_tool']
```

两点留意：

1. 扫出来的是 **13 个**（与源文件的实测结论一致）、而且**按字母序**而不是定义顺序 ——
   `dir()` 就是这个行为，别把它的顺序当成「定义顺序」来依赖；
2. `<module 'tools_jxsd'>` 是按原模块边界造出来的代理模块，
   `call_tool` 被挡在外面了 —— 这一点下面会拿实测数据对照。

### 2.6 `list_tools()` 与 `call_tool()`：课案原文 + 两层保护

课案原文只有两行：

```python
def list_tools():
    """列出tools.py中所有的工具"""
    return [{"工具名": func, "工具描述": getattr(tools, func).__doc__}
            for func in dir(tools) if func.endswith("_tool")]

def call_tool(tool_name, *args, **kwargs):
    """调用工具"""
    return getattr(tools, tool_name)(*args, **kwargs)
```

完整版在**保持课案行为**的前提下多做了三件事：

| 改动 | 原因 |
|---|---|
| `list_tools` 多了 `"参数"` 键 | 课案把 `parameters` 写死成 `{a: number, b: number}`，只能给 4 个数学工具用 |
| `list_tools(only=...)` 可选过滤 | 让「数学四则」和「参数示范」两套工具能分开取 |
| `call_tool` 加**名字白名单 + 执行兜底** | 参数是**模型生成的、不可信**：名字可能不存在，参数可能非法（如 `div_tool(1, 0)`） |

那两层保护不是「掩盖错误」，而是 Agent **自愈能力**的来源：
把错误当成「工具执行结果」回传给模型，模型下一轮往往能自己纠正。

⚠️ 注意 `"工具名" / "工具描述" / "参数"` 这几个**中文键名是课案的习惯**，只在本项目内部用；
发给模型前必须映射成 `name` / `description` / `parameters`（2.9 就是干这个的）。

In [ ]:
# ================================================================
# 九、课案的 list_tools / call_tool（import 的模块名换成 tools_jxsd）
# ================================================================
def list_tools(only: list[str] | None = None) -> list[dict]:
    """
    列出 tools_jxsd.py 中所有的工具

    :param only: 可选，只列这几个工具名；不传 = 列出全部（课案原行为）
    :return: [{"工具名": ..., "工具描述": ..., "参数": ...}, ...]

    三点说明：
        1. 课案用 dir() + endswith("_tool") 扫描模块，这正是**命名后缀约定**的意义——
           只要函数叫 xxx_tool，就会被自动收集，不需要手工维护清单；
        2. "参数" 这个键是课案原文没有、但主循环需要的：课案在 agent.py 里把
           parameters 写死成 {"a": number, "b": number}，那是给 4 个数学工具专用的；
           本文件工具变多了，所以改成按工具名查表（见下方的 TOOL_PARAMETERS）。
        3. "工具描述" 取的是整个 __doc__（课案原样），因此连 :param / :return 那几行
           也一起发给了模型。实战里可以只取第一行做精简描述，这里保持课案原样子，
           方便对照「模型实际看到的是什么」。
    """
    names = [func for func in dir(tools_jxsd) if func.endswith("_tool")]
    if only is not None:
        # 排序保证顺序稳定：dir() 返回的列表是按字母序的，过滤后仍然是字母序
        names = [func for func in names if func in only]
    return [
        {
            "工具名": name,
            "工具描述": getattr(tools_jxsd, name).__doc__,
            "参数": TOOL_PARAMETERS.get(name, EMPTY_PARAMETERS),
        }
        for name in names
    ]


def call_tool(tool_name, *args, **kwargs):
    """
    调用工具：按名字从 tools_jxsd 模块里取出真函数并执行

    课案原文是 `return getattr(tools, tool_name)(*args, **kwargs)`，一行搞定。
    本文件多包了两层保护，因为**参数是模型生成的，不可信**：
        1. 名字白名单：模型可能报出一个不存在的函数名，getattr 会抛 AttributeError；
           更危险的是，不设白名单时模型若能控制名字，就可能 getattr 到 os.system 这类函数；
        2. 执行兜底：模型可能给出 div_tool(a=1, b=0) 这种非法参数。
    这两类错误都不该让主循环崩掉——把错误当成「工具执行结果」回传给模型，
    模型下一轮往往能自己纠正（这正是 Agent 自愈能力的来源）。
    """
    func = tools_jxsd.TOOL_REGISTRY.get(tool_name)
    if func is None:
        return f"没有名为 {tool_name} 的工具"
    try:
        return func(*args, **kwargs)
    except Exception as exc:                      # noqa: BLE001 —— 故意兜住全部异常，回传给模型
        return f"工具 {tool_name} 执行出错：{type(exc).__name__}: {exc}"

`call_tool` 定义完了，正好可以把上面那段警告**用实测数据钉死**：
同一个「以 `_tool` 结尾」的筛选条件，套在两个不同的命名空间上，结果差一个 `call_tool`。

In [ ]:
# 反面对照：不隔离模块边界，分发器自己会被当成「工具」
print("扫 tools_jxsd（还原了模块边界）：", [n for n in dir(tools_jxsd) if n.endswith("_tool")])
print("扫 notebook __main__（未隔离）    ：", [n for n in dir(sys.modules[__name__]) if n.endswith("_tool")])

### 预期输出

```text
扫 tools_jxsd（还原了模块边界）： ['add_tool', 'calc_price_tool', 'check_stock_tool', 'convert_temperature_tool', 'create_user_tool', 'describe_user_tool', 'div_tool', 'format_username_tool', 'mul_tool', 'set_nickname_tool', 'sub_tool', 'summarize_tags_tool', 'toggle_feature_tool']
扫 notebook __main__（未隔离）    ： ['add_tool', 'calc_price_tool', 'call_tool', 'check_stock_tool', 'convert_temperature_tool', 'create_user_tool', 'describe_user_tool', 'div_tool', 'format_username_tool', 'mul_tool', 'set_nickname_tool', 'sub_tool', 'summarize_tags_tool', 'toggle_feature_tool']
```

第二行比第一行**多一个 `call_tool`** —— 它就夹在 `calc_price_tool` 和 `check_stock_tool` 之间。
这不是「假设」，是本节改造过程中真实踩到、又真实修掉的一个坑：
最初把 notebook 的 `__main__` 直接注册成 `tools_jxsd`，跑出来的工具数就是 **14**，
多出来的那个正是分发器本身，会跟着 `TOOLS` 一起发给模型。

教训：**在「靠命名约定扫描」的模块里，别把名字收尾是 `_tool` 的东西放进去** ——
源文件 `tools_jxsd.py` 的注释早就警告过这一条（「任何以 `_tool` 结尾的顶层名字
都会被自动当成工具发出去」），平时它只坑随手加的辅助函数，
而**合并模块时会直接坑到分发器本身**。

`list_tools` 里引用的 `TOOL_PARAMETERS` / `EMPTY_PARAMETERS` 此刻还没定义 ——
没关系，它们是在**函数被调用时**才去全局命名空间里找的。
这两个名字马上就要在 2.8 里定义，而 `list_tools()` 第一次被调用是在 2.9。

### 2.7 课案《参数类型》小节的 8 个 Schema 片段

先明确一个最容易混的点：**下面这些片段描述的是「一个参数」，不是整套 `parameters`**。

它们里面的 `"name"` 键**不属于 JSON Schema 标准** —— 那是课案为了讲清
「这个片段对应哪个参数名」而人工加上的标记。真正的含义是：

| 键 | 含义 | 拼进 `parameters` 时的去向 |
|---|---|---|
| `name` | 参数名（**非标准键**） | 变成 `properties` 的**键名**，自身要删掉 |
| `type` | JSON Schema 类型关键字 | 原样保留 |
| `description` | 写给模型看的自然语言说明，**最影响准确率** | 原样保留 |
| `items` / `properties` / `enum` | 数组元素类型 / 对象字段 / 枚举取值 | 原样保留 |

⚠️ 组装时**必须把 `name` 去掉**（2.8 的 `build_parameters` 就干这个），
否则模型会多填一个名叫 `name` 的字段。

先看四种基础类型：

In [ ]:
# ================================================================
# 十、课案《参数类型》小节的 Schema 片段（原文保留）
# ================================================================
# ---------- 1. string —— 字符串类型 ----------
SCHEMA_STRING = {
    "name": "username",
    "type": "string",
    "description": "用户名",
}

# ---------- 2. number —— 数字类型（整数或浮点数） ----------
SCHEMA_NUMBER = {
    "name": "price",
    "type": "number",
    "description": "商品价格",
}

# ---------- 3. integer —— 整数类型 ----------
SCHEMA_INTEGER = {
    "name": "count",
    "type": "integer",
    "description": "商品数量",
}

# ---------- 4. boolean —— 布尔值 ----------
SCHEMA_BOOLEAN = {
    "name": "is_enabled",
    "type": "boolean",
    "description": "是否启用",
}

参数名用 `is_` / `has_` 开头是惯例：模型看到 `is_enabled`
就知道该填 `true` 或 `false`，而不是填「是」。

接着是 `array`（**必须**带 `items`）和 `object`（**必须**带 `properties`）：

In [ ]:
# ---------- 5. array —— 数组类型（需指定 items 元素类型） ----------
SCHEMA_ARRAY = {
    "name": "tags",
    "type": "array",
    "items": {"type": "string"},
    "description": "标签列表",
}

# ---------- 6. object —— 对象类型（需指定 properties） ----------
SCHEMA_OBJECT = {
    "name": "user",
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "age": {"type": "integer"},
    },
    "description": "用户信息",
}

`array` 的 `items` 若换成对象数组，就写成
`"items": {"type": "object", "properties": {...}}`；
对象里的字段再套一个 `type: object` 就是**嵌套对象**，见本节的 `SCHEMA_OBJECT_NESTED`。

最后是 `null`（可选字段的标准写法）与课案《8. 完整示例》的 `create_user`：

In [ ]:
# ---------- 7. null —— 空值（常用于可选字段） ----------
SCHEMA_NULL = {
    "name": "optional_field",
    "type": ["string", "null"],   # 可以为 string 或 null
    "description": "可选字段",
}

# ---------- 8. 完整示例 —— 课案 3738-3761 的 create_user ----------
# 注意：这个片段跟前 7 个不同，它是一个**完整的 tools 元素**（含 type/function 两层），
# 可以直接塞进 client.chat.completions.create(tools=[SCHEMA_CREATE_USER]) 里用。
SCHEMA_CREATE_USER = {
    "type": "function",
    "function": {
        "name": "create_user",
        "description": "创建用户",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "用户名"},
                "age": {"type": "integer", "description": "年龄"},
                "email": {"type": "string", "description": "邮箱"},
                "is_active": {"type": "boolean", "description": "是否激活"},
                "tags": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "标签",
                },
            },
            "required": ["name", "email"],
        },
    },
}

`type` 写成**列表**表示多选一：`["string", "null"]` = 要么字符串、要么 null。
这是 JSON Schema 表达「可选字段」的标准写法 ——
OpenAI 严格模式下，可选参数就是靠它实现的：字段仍然出现在 `required` 里，但允许模型传 `null`。

`SCHEMA_CREATE_USER` 的 `required` 只有 `["name", "email"]`，`age` / `is_active` / `tags` 可省。
**必填与可选的划分很关键**：全必填会让模型硬编造参数，全可选又容易漏填。

再补两个课案里没写全的片段：**enum** 和**嵌套对象**。

In [ ]:
# ---------- 补充 A：enum 枚举（课案 agent.py 里那处 enum 片段的正确写法） ----------
SCHEMA_ENUM_VALUE = {
    "name": "value",
    "type": "number",
    "description": "温度数值",
}
SCHEMA_ENUM_UNIT = {
    "name": "unit",
    "type": "string",
    "enum": ["celsius", "fahrenheit"],
    "description": "传入数值的单位",
}

# ---------- 补充 B：嵌套对象（object 里再套 object） ----------
SCHEMA_OBJECT_NESTED = {
    "type": "object",
    "properties": {
        "name": {"type": "string", "description": "用户名"},
        "age": {"type": "integer", "description": "年龄"},
        "address": {
            "type": "object",
            "description": "地址（对象里再套一个对象）",
            "properties": {
                "city": {"type": "string", "description": "城市"},
                "zip": {"type": "string", "description": "邮编"},
            },
            "required": ["city", "zip"],
            # additionalProperties：禁止出现 Schema 里没声明的字段。
            # 它和 strict 是一对：开了 strict，每个 object 都得写上这一句，
            # 否则模型仍可能自由发挥出多余字段。
            "additionalProperties": False,
        },
    },
    "required": ["name", "age", "address"],
    "additionalProperties": False,
}

⚠️ **课案 `agent.py` 里那处 enum 是个笔误**，值得单独记一笔：

```python
"b": {"type": "number", "enum": ["celsius", "fahrenheit"]}
```

枚举值是字符串，类型却写了 `number`。正确写法是
**`type: "string"` + `enum: [...]`**，也就是上面 `SCHEMA_ENUM_UNIT` 的样子。

### 2.8 `build_parameters()`：把「片段」组装成「完整 parameters」

组装规则只有一条：**片段的 `name` → `properties` 的键；片段其余字段 → 该键的值**。

```text
SCHEMA_STRING = {"name": "username", "type": "string", "description": "用户名"}
                         ↓ 组装
"properties": {"username": {"type": "string", "description": "用户名"}}
```

顺带三个说明：

- `EMPTY_PARAMETERS` 是**没写参数的工具有没有兜底**用的（一个不接任何参数的 object）；
- `SCHEMA_MATH_A` / `SCHEMA_MATH_B` 就是课案 `agent.py` 给 4 个数学工具写死的那份
  `{a: number, b: number}`，这里把它还原成两个可复用的片段；
- `required` **默认全必填** —— 这是 OpenAI 严格模式的硬性要求，见「常见坑」。

In [ ]:
# ================================================================
# 十一、把「片段」组装成「完整的 parameters」
# ================================================================
def build_parameters(*fields: dict, required: list[str] | None = None) -> dict:
    """
    把若干「参数片段」组装成一个完整的 parameters（JSON Schema object）

    :param fields: 参数片段，例如 SCHEMA_STRING、SCHEMA_NUMBER
    :param required: 必填字段名列表；不传 = 所有字段都必填
    :return: {"type": "object", "properties": {...}, "required": [...], "additionalProperties": False}
    """
    properties = {}
    for field in fields:
        name = field["name"]
        # 去掉非标准的 name 键，其余字段原样保留（type / description / enum / items / properties……）
        properties[name] = {key: value for key, value in field.items() if key != "name"}
    return {
        "type": "object",
        "properties": properties,
        # 默认全必填：OpenAI 严格模式要求 required 覆盖 properties 的全部字段，
        # 「可选」改用 type: ["string", "null"] 表达（见 SCHEMA_NULL）
        "required": list(required) if required is not None else list(properties),
        "additionalProperties": False,
    }


# 没有描述的工具走这个兜底 Schema：一个不接任何参数的 object
EMPTY_PARAMETERS = {"type": "object", "properties": {}, "required": [], "additionalProperties": False}

# 课案 agent.py 给 4 个数学工具写死的参数：a、b 都是 number
SCHEMA_MATH_A = {"name": "a", "type": "number", "description": "第一个数字"}
SCHEMA_MATH_B = {"name": "b", "type": "number", "description": "第二个数字"}

# 工具名 → 完整 parameters。键必须和 tools_jxsd.py 里的函数名严格一致
TOOL_PARAMETERS = {
    # 数学四则：对应课案 agent.py 里写死的 {"a": number, "b": number}
    **{name: build_parameters(SCHEMA_MATH_A, SCHEMA_MATH_B) for name in tools_jxsd.MATH_TOOL_NAMES},
    # 课案《参数类型》8 种类型，逐个落到 tools_jxsd.py 的真函数上
    "format_username_tool": build_parameters(SCHEMA_STRING),
    "calc_price_tool": build_parameters(SCHEMA_NUMBER),
    "check_stock_tool": build_parameters(SCHEMA_INTEGER),
    "toggle_feature_tool": build_parameters(SCHEMA_BOOLEAN),
    "summarize_tags_tool": build_parameters(SCHEMA_ARRAY),
    "describe_user_tool": SCHEMA_OBJECT_NESTED,      # 对象 + 嵌套对象
    "set_nickname_tool": build_parameters(SCHEMA_NULL),
    "convert_temperature_tool": build_parameters(SCHEMA_ENUM_VALUE, SCHEMA_ENUM_UNIT),
    # 课案《8. 完整示例》的 create_user：直接取它 parameters 那一段（多类型组合）
    "create_user_tool": SCHEMA_CREATE_USER["function"]["parameters"],
}

⚠️ `TOOL_PARAMETERS` 的键**必须和 `tools_jxsd.py` 里的函数名严格一致**；
对不上的话，`list_tools()` 会静默回落到 `EMPTY_PARAMETERS`（工具变成「不接参数」），
而不是报错 —— 这是最容易查漏的一类问题。

### 2.9 `TOOLS`：最终发给模型的那份声明

课案 `agent.py` 用的是**列表推导式动态生成**（关键点说明 1：「动态生成 tools」），
本节的推导式骨架和课案**完全一致**，只把写死的 `parameters` 换成 `tool["参数"]`。

另外注意那个 `"strict": True`：课案原文里它被写进了 `parameters` 内部
（`parameters` 那行的 `}` 后漏了逗号，**运行前就会 `SyntaxError`**），
本节按正确的层级（`function` 的兄弟键）生成，位置见下面输出。

In [ ]:
# ================================================================
# 十二、多工具声明：最终喂给 client.chat.completions.create(tools=...) 的列表
# ================================================================
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": tool["工具名"],
            "description": tool["工具描述"],
            "parameters": tool["参数"],
            # strict：课案原文写着 True。开启后模型必须生成合法 JSON（结构化输出约束），
            # 但要求每个 object 都带 additionalProperties: False，且 required 覆盖所有字段。
            # 本文件的 Schema 已经满足这两条，所以原样保持 True；
            # 若换成不支持 strict 的服务（部分国产中转不认这个字段），把它改成 False 即可。
            "strict": True,
        },
    }
    for tool in list_tools()
]

这里有个**前文埋的伏笔要说明**：`TOOLS` 和 `TOOL_REGISTRY` 这两个名字，
在课案原版（1.1 / 1.2）里已经各出现过一次 —— 本节第二次定义会**覆盖**它们。

所以 1.3 的演示必须放在 1.1 / 1.2 之后、本节之前，顺序不能调换。
覆盖后 `TOOL_REGISTRY` 是 13 个工具的版本、`TOOLS` 是 13 份描述，
下一课 `02_三种Agent实现对比.ipynb` import 到的就是这一份（完整的那个）。

### 2.10 本地自测：打印全部 Schema，并演示取工具 / 调工具

同样是把 `tool_desc_jxsd.py` 原 `__main__` 里的内容**顶格照抄**。
第 ④ 节那三条 `call_tool` 演示分别打到三条不同路径，
**它们就是「两层保护都在工作」的证据**。

In [ ]:
# ================================================================
# 十三、本地自测：打印全部 Schema，并演示取工具 / 调工具（原 __main__ 顶格）
# ================================================================
print("=" * 62)
print("一、课案《参数类型》的 8 个 Schema 片段（原样打印，逐个对照注释看）")
print("=" * 62)
fragments = [
    ("1. string", SCHEMA_STRING),
    ("2. number", SCHEMA_NUMBER),
    ("3. integer", SCHEMA_INTEGER),
    ("4. boolean", SCHEMA_BOOLEAN),
    ("5. array", SCHEMA_ARRAY),
    ("6. object", SCHEMA_OBJECT),
    ("7. null", SCHEMA_NULL),
    ("补充 A. enum", SCHEMA_ENUM_UNIT),
]
for title, fragment in fragments:
    # ensure_ascii=False → 中文原样输出（默认 True 会打成 \uXXXX，看不懂）
    print(f"{title}：{json.dumps(fragment, ensure_ascii=False)}")

print()
print("8. 完整示例（create_user，整段 tools 元素）：")
print(json.dumps(SCHEMA_CREATE_USER, ensure_ascii=False, indent=2))

print()
print("=" * 62)
print("二、list_tools()：把片段组装成每个工具的 parameters")
print("=" * 62)
for tool in list_tools():
    props = list(tool["参数"]["properties"])
    print(f"{tool['工具名']:<26} 参数={props} 必填={tool['参数']['required']}")

print()
print("=" * 62)
print("三、TOOLS 多工具声明（最终发给模型的东西）")
print("=" * 62)
print(f"共 {len(TOOLS)} 个工具：{[t['function']['name'] for t in TOOLS]}")
print("其中一个工具的完整声明（describe_user_tool，含嵌套对象）：")
sample = next(t for t in TOOLS if t["function"]["name"] == "describe_user_tool")
print(json.dumps(sample, ensure_ascii=False, indent=2)[:600] + " ...")

print()
print("=" * 62)
print("四、call_tool()：按名字分发到真函数（含模型乱传参数时的兜底）")
print("=" * 62)
print(f"正常调用   call_tool('mul_tool', a=4, b=6)  -> {call_tool('mul_tool', a=4, b=6)}")
print(f"名字写错   call_tool('mul_tool_typo', a=1)  -> {call_tool('mul_tool_typo', a=1)}")
print(f"参数非法   call_tool('div_tool', a=1, b=0)  -> {call_tool('div_tool', a=1, b=0)}")
print()
print("下一步：agent_openai_jxsd.py 把这些描述发给模型，跑真正的 Function Call 主循环。")

### 预期输出

```text
==============================================================
一、课案《参数类型》的 8 个 Schema 片段（原样打印，逐个对照注释看）
==============================================================
1. string：{"name": "username", "type": "string", "description": "用户名"}
2. number：{"name": "price", "type": "number", "description": "商品价格"}
3. integer：{"name": "count", "type": "integer", "description": "商品数量"}
4. boolean：{"name": "is_enabled", "type": "boolean", "description": "是否启用"}
5. array：{"name": "tags", "type": "array", "items": {"type": "string"}, "description": "标签列表"}
6. object：{"name": "user", "type": "object", "properties": {"name": {"type": "string"}, "age": {"type": "integer"}}, "description": "用户信息"}
7. null：{"name": "optional_field", "type": ["string", "null"], "description": "可选字段"}
补充 A. enum：{"name": "unit", "type": "string", "enum": ["celsius", "fahrenheit"], "description": "传入数值的单位"}

8. 完整示例（create_user，整段 tools 元素）：
{
  "type": "function",
  "function": {
    "name": "create_user",
    "description": "创建用户",
    "parameters": {
      "type": "object",
      "properties": {
        "name": {
          "type": "string",
          "description": "用户名"
        },
        "age": {
          "type": "integer",
          "description": "年龄"
        },
        "email": {
          "type": "string",
          "description": "邮箱"
        },
        "is_active": {
          "type": "boolean",
          "description": "是否激活"
        },
        "tags": {
          "type": "array",
          "items": {
            "type": "string"
          },
          "description": "标签"
        }
      },
      "required": [
        "name",
        "email"
      ]
    }
  }
}

==============================================================
二、list_tools()：把片段组装成每个工具的 parameters
==============================================================
add_tool                   参数=['a', 'b'] 必填=['a', 'b']
calc_price_tool            参数=['price'] 必填=['price']
check_stock_tool           参数=['count'] 必填=['count']
convert_temperature_tool   参数=['value', 'unit'] 必填=['value', 'unit']
create_user_tool           参数=['name', 'age', 'email', 'is_active', 'tags'] 必填=['name', 'email']
describe_user_tool         参数=['name', 'age', 'address'] 必填=['name', 'age', 'address']
div_tool                   参数=['a', 'b'] 必填=['a', 'b']
format_username_tool       参数=['username'] 必填=['username']
mul_tool                   参数=['a', 'b'] 必填=['a', 'b']
set_nickname_tool          参数=['optional_field'] 必填=['optional_field']
sub_tool                   参数=['a', 'b'] 必填=['a', 'b']
summarize_tags_tool        参数=['tags'] 必填=['tags']
toggle_feature_tool        参数=['is_enabled'] 必填=['is_enabled']

==============================================================
三、TOOLS 多工具声明（最终发给模型的东西）
==============================================================
共 13 个工具：['add_tool', 'calc_price_tool', 'check_stock_tool', 'convert_temperature_tool', 'create_user_tool', 'describe_user_tool', 'div_tool', 'format_username_tool', 'mul_tool', 'set_nickname_tool', 'sub_tool', 'summarize_tags_tool', 'toggle_feature_tool']
其中一个工具的完整声明（describe_user_tool，含嵌套对象）：
{
  "type": "function",
  "function": {
    "name": "describe_user_tool",
    "description": "\n    把用户信息渲染成一句话（user 里可以带嵌套的 address 对象）\n    :param user: 用户信息对象，形如 {\"name\": \"张三\", \"age\": 28,\n                 \"address\": {\"city\": \"南昌\", \"zip\": \"330000\"}}\n    :return: 用户描述\n    ",
    "parameters": {
      "type": "object",
      "properties": {
        "name": {
          "type": "string",
          "description": "用户名"
        },
        "age": {
          "type": "integer",
          "description": "年龄"
        },
        "address": {
          "type": "object",
          "des ...

==============================================================
四、call_tool()：按名字分发到真函数（含模型乱传参数时的兜底）
==============================================================
正常调用   call_tool('mul_tool', a=4, b=6)  -> 24
名字写错   call_tool('mul_tool_typo', a=1)  -> 没有名为 mul_tool_typo 的工具
参数非法   call_tool('div_tool', a=1, b=0)  -> 工具 div_tool 执行出错：ZeroDivisionError: division by zero

下一步：agent_openai_jxsd.py 把这些描述发给模型，跑真正的 Function Call 主循环。
```

**第 ④ 节这三行是本节最实用的部分**，它们分别命中三条路径：

| 调用 | 命中路径 | 说明 |
|---|---|---|
| `mul_tool(a=4, b=6)` | 正常执行 | 返回真值 `24` |
| `mul_tool_typo` | **名字白名单** | 名字不在 `TOOL_REGISTRY` 里 → 返回中文提示而不是崩掉 |
| `div_tool(a=1, b=0)` | **执行兜底** | 真函数抛了 `ZeroDivisionError` → 被 `try/except` 兜住，变成一句可回传给模型的错误描述 |

另外留意第 二 节打印的**顺序是字母序**而不是定义顺序 —— 这是 `dir()` 的行为，
别把它当成「定义顺序」来依赖（想固定顺序就自己排序）。

第 三 节最后那行 `"des ...` 不是打错了：源文件那句是
`print(json.dumps(sample, ensure_ascii=False, indent=2)[:600] + " ...")`，
**在 600 个字符处硬截断**再拼上 ` ...`，所以末尾正好断在一个键名中间。
这也顺带说明了 `description` 里塞的是**整个 docstring**（含 `:param` / `:return`）
—— 它们会一起发给模型、一起计费。

### 2.11 同一件事，三种写法

到这里「工具 = 函数 + 描述」这条主线就走完了。下一课会看到：
**换框架并不改变这条主线，只是有人替你把描述生成出来、把主循环写好**。

| | 工具本体 | 描述从哪来 | 谁写主循环 |
|---|---|---|---|
| 本课（原生 OpenAI SDK 路线） | 普通 `def` | **手写** `TOOLS` 列表 | 你（下一课手写 while 循环） |
| LangChain | `@tool` 装饰的函数 | 装饰器读**形参名 + 注解 + docstring** 自动生成 | `create_agent` |
| DeepAgents | 普通 `def` | 自动生成 | `create_deep_agent` |

所以 2.2 里那些类型注解不是装饰品 —— 在 LangChain 那条线上，
**注解就是 `properties` 里的 `type`，docstring 就是 `description`**。
官方链接见文末「工具（Tools）」一页。

## 小结

- **工具是两件东西**：给程序看的**函数**（真正干活）+ 给模型看的**描述**（JSON Schema）；
- **描述三要素**：`name`（调谁）/ `description`（何时调）/ `parameters`（怎么填），
  其中 `description` 最影响调用准确率 —— 模型看不到函数代码；
- **对齐键是函数名**：`tool_call.name` ↔ `TOOL_REGISTRY` 的键，两边名字对不上就是最经典的故障；
- **注册表是白名单**，同时兼有安全意义（模型不能让你 `getattr` 到任意函数）；
- **类型写得越准，模型乱填越少**：`number` vs `integer`、`enum` 限取值、
  `items` 限数组元素、`required` 划分必填与可选；
- **分发要兜底**：名字不存在、参数非法都不该让主循环崩掉 ——
  把错误当成「工具结果」回传给模型，这是 Agent 自愈能力的来源。

## 常见坑

1. **函数名对不上描述名**。现象是「模型说要调 `get_weather`，程序回一句
   『没有名为 get_weather 的工具』」。排查顺序：先比对两边名字，再看参数。
2. **`TOOL_PARAMETERS` 漏了某个工具**。它**不报错**，只是静默回落到
   `EMPTY_PARAMETERS`，工具变成「不接参数」—— 模型填的参数会被丢掉。
3. **JSON Schema 里没有 `name` 这个关键字**。本课片段里的 `"name"` 是课案加的标记，
   组装时必须去掉（`build_parameters` 干的就是这件事）；
   忘了去掉，模型会多填一个叫 `name` 的字段。
4. **中文键名不能直接发给模型**。`"工具名" / "工具描述" / "参数"` 是本项目的内部约定，
   发出去之前必须映射成 `name` / `description` / `parameters`（见 2.9）。
5. **`strict: True` 有硬性配套条件**：每个 object 都要写 `additionalProperties: False`，
   且 `required` 必须覆盖 `properties` 的全部字段。少一条，服务端直接返回 400，
   而报错信息通常只说「schema 不合法」，**不会告诉你是哪一条**。
6. **`div_tool(1, 0)` 会抛 `ZeroDivisionError`**（课案原版没有除零保护）。
   这里是**故意保持一致**的：让「课案原文的写法会怎样出错」在第 ④ 节真的看到，
   而不是被提前掩盖。生产环境的保护放在 `call_tool` 的 `try/except` 里。
7. **对象参数要用 `.get` 兜底**。模型偶尔会漏字段，
   `user["age"]` 会 `KeyError` 把整个主循环炸掉。
8. **`list[str] | None` 需要 Python 3.10+**。老环境上要改回 `Optional[List[str]]`；
   本机 `.venv` 是 3.12，满足。
9. **`__doc__` 是整个 docstring**，连 `:param` / `:return` 那几行也会发给模型、**一起计费**。
   本节保持课案原样，是为了让你看到「模型实际收到了什么」；
   实战里通常只取第一行做精简描述。
10. **别让名字以 `_tool` 结尾的东西进到工具模块里**：`dir()` + `endswith("_tool")`
   会把它们一起当成工具发出去。本节就真实踩过这一条 ——
   `call_tool` 恰好以 `_tool` 结尾，合并模块后它一度成为「第 14 个工具」（见 2.6 的实测对照）。
   平时它坑的是随手加的辅助函数，合并模块时坑的是分发器本身。

## 官方链接

- 工具（Tools）—— `@tool` 装饰器、Schema 自动生成、参数类型：
  <https://docs.langchain.com/oss/python/langchain/tools>
- 智能体（Agents）—— `create_agent` 如何替你跑主循环（下一课的重点）：
  <https://docs.langchain.com/oss/python/langchain/agents>
- JSON Schema 规范（本课所有 `type` / `properties` / `required` / `items` / `enum`
  的出处）：<https://json-schema.org/understanding-json-schema/reference>